# MP2 Part 1 - Commit retrieval (netid: gkhot)

Retrieves every commit of my 10 assigned projects from World of Code (WoC, snapshot V2412, data up to ~Jan 2025) and the stars / forks / last commit date from the GitHub REST API.

In [1]:
# !python -m pip install -U python-woc pandas matplotlib tqdm requests

In [2]:
import os, time, json
import pandas as pd
import requests
from tqdm import tqdm
from woc.remote import WocMapsRemote

netid = 'gkhot'
woc = WocMapsRemote(base_url="https://worldofcode.org/api/")

assign = pd.read_csv('net2prj.csv')
assign = assign[assign['netID'] == netid].reset_index(drop=True)
projects = assign['WoC'].tolist()
assign

/Users/gururajkhot/Mini-Project-2/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


,netID,WoC,GH
0,gkhot,alanwilter_acpype,https://github.com/alanwilter/acpype
1,gkhot,sheffieldml_gpy,https://github.com/SheffieldML/GPy
2,gkhot,rrrlw_ripserr,https://github.com/rrrlw/ripserr
3,gkhot,dakep_pense-rpkg,https://github.com/dakep/pense-rpkg
4,gkhot,declaredesign_estimatr,https://github.com/DeclareDesign/estimatr
5,gkhot,igsio_openigtlinkio,https://github.com/IGSIO/OpenIGTLinkIO
6,gkhot,aura-healthcare_hrv-analysis,https://github.com/Aura-healthcare/hrv-analysis
7,gkhot,juliaopt_cplex.jl,https://github.com/JuliaOpt/CPLEX.jl
8,gkhot,rwth-i6_returnn,https://github.com/rwth-i6/returnn
9,gkhot,ipeagit_geobr,https://github.com/ipeaGIT/geobr


## Step 1: commit sha1s per project (p2c map)

In [3]:
os.makedirs('cache', exist_ok=True)
if os.path.exists('cache/df_commits.csv'):  # retrieved earlier, reuse instead of hitting WoC again
    df = pd.read_csv('cache/df_commits.csv')
else:
    list_df_commits = []
    for prj in projects:
        commits = woc.get_values('p2c', prj)
        d = pd.DataFrame(commits, columns=['sha1'])
        d['project'] = prj
        list_df_commits.append(d)
        time.sleep(1)
    df = pd.concat(list_df_commits, ignore_index=True)
    df.to_csv('cache/df_commits.csv', index=False)
df.groupby('project').size()

project
alanwilter_acpype                 919
aura-healthcare_hrv-analysis      177
dakep_pense-rpkg                  754
declaredesign_estimatr           1423
igsio_openigtlinkio               692
ipeagit_geobr                    4942
juliaopt_cplex.jl                1159
rrrlw_ripserr                     429
rwth-i6_returnn                 16730
sheffieldml_gpy                  6877
dtype: int64

## Step 2: commit content in batches
Chunks of 40 commits (README: keep chunks under 50) with a 2 s wait between requests.

**Fallback:** worldofcode.org repeatedly went offline (connection refused) during my retrieval on 2026-10-07. For any project that could not be fully retrieved from WoC, commits are taken from the project's own git history (all branches), restricted to author times before 2025-02-01 to match the WoC V2412 snapshot. The `source` column of `data_source` below records which projects used WoC and which used git. Re-running this cell retries WoC for the git-sourced projects.

In [4]:
CHUNK = 40
WOC_CUTOFF = 1738368000  # 2025-02-01 00:00 UTC, end of the WoC V2412 snapshot

def woc_up():
    try:
        requests.get('https://worldofcode.org/', timeout=10)
        return True
    except requests.RequestException:
        return False

def fetch_batch(shas, tries=4):
    global woc
    for attempt in range(tries):
        try:
            res, err = woc.get_values_many('commit.tch', shas)
            return {k: v[0] for k, v in res.items()}, err
        except Exception as e:
            print('  retry', attempt + 1, type(e).__name__, str(e)[:80])
            time.sleep(15 * (attempt + 1))
            # a timed-out request leaves python-woc's event loop in a bad state, so start a fresh client
            woc = WocMapsRemote(base_url="https://worldofcode.org/api/")
    raise ConnectionError('WoC batch failed repeatedly')

def woc_project_commits(prj):
    shas = df.loc[df['project'] == prj, 'sha1'].tolist()
    rows, errors = [], {}
    for i in tqdm(range(0, len(shas), CHUNK), desc=prj):
        res, err = fetch_batch(shas[i:i + CHUNK])
        errors.update(err or {})
        for sha, c in res.items():
            rows.append({'project': prj, 'commit': sha,
                         'author': c[2][0], 'author_time': int(c[2][1]),
                         'committer': c[3][0], 'committer_time': int(c[3][1]),
                         'message': c[4]})
        time.sleep(2)
    if errors:
        print(f'{prj}: {len(errors)} errors', list(errors.items())[:3])
    return pd.DataFrame(rows)

def git_project_commits(prj, url):
    import subprocess
    path = f'../ghclones/{prj}.git'
    if not os.path.exists(path):
        subprocess.run(['git', 'clone', '-q', '--bare', '--filter=blob:none', url, path], check=True)
    US, RS = '\x1f', '\x1e'
    log = subprocess.run(['git', '-C', path, 'log', '--all',
                          f'--format=%H{US}%an <%ae>{US}%at{US}%cn <%ce>{US}%ct{US}%B{RS}'],
                         capture_output=True, text=True, errors='replace', check=True).stdout
    rows = [r.strip('\n').split(US, 5) for r in log.split(RS) if r.strip()]
    out = pd.DataFrame(rows, columns=['commit', 'author', 'author_time', 'committer', 'committer_time', 'message'])
    out[['author_time', 'committer_time']] = out[['author_time', 'committer_time']].astype(int)
    out.insert(0, 'project', prj)
    return out[out['author_time'] < WOC_CUTOFF]

parts, sources = [], {}
for prj, url in zip(assign['WoC'], assign['GH']):
    woc_cache, git_cache = f'cache/{prj}_commit_data.csv', f'cache/{prj}_git_commit_data.csv'
    if os.path.exists(woc_cache):
        parts.append(pd.read_csv(woc_cache)); sources[prj] = 'WoC'
        continue
    if woc_up():
        try:
            d = woc_project_commits(prj)
            d.to_csv(woc_cache, index=False)
            parts.append(d); sources[prj] = 'WoC'
            continue
        except ConnectionError as e:
            print(f'{prj}: WoC failed ({e}), falling back to git history')
    else:
        print(f'{prj}: worldofcode.org unreachable, using git history')
    d = git_project_commits(prj, url)
    d.to_csv(git_cache, index=False)
    parts.append(d); sources[prj] = 'git (WoC unavailable)'

commit_data = pd.concat(parts, ignore_index=True)
data_source = pd.Series(sources, name='source').rename_axis('project').reset_index()
data_source.to_csv('cache/data_source.csv', index=False)
data_source

alanwilter_acpype: worldofcode.org unreachable, using git history
sheffieldml_gpy: worldofcode.org unreachable, using git history


rrrlw_ripserr: worldofcode.org unreachable, using git history


dakep_pense-rpkg: worldofcode.org unreachable, using git history


declaredesign_estimatr: worldofcode.org unreachable, using git history


igsio_openigtlinkio: worldofcode.org unreachable, using git history


aura-healthcare_hrv-analysis: worldofcode.org unreachable, using git history


juliaopt_cplex.jl: worldofcode.org unreachable, using git history


rwth-i6_returnn: worldofcode.org unreachable, using git history


ipeagit_geobr: worldofcode.org unreachable, using git history


,project,source
0,alanwilter_acpype,git (WoC unavailable)
1,sheffieldml_gpy,git (WoC unavailable)
2,rrrlw_ripserr,git (WoC unavailable)
3,dakep_pense-rpkg,git (WoC unavailable)
4,declaredesign_estimatr,git (WoC unavailable)
5,igsio_openigtlinkio,git (WoC unavailable)
6,aura-healthcare_hrv-analysis,git (WoC unavailable)
7,juliaopt_cplex.jl,git (WoC unavailable)
8,rwth-i6_returnn,git (WoC unavailable)
9,ipeagit_geobr,git (WoC unavailable)


## Validation: did we get every commit listed by p2c?
Only meaningful for WoC-sourced projects; git-sourced counts differ because WoC's p2c also includes commits WoC attributes to the project from forks/copies.

In [5]:
check = pd.DataFrame({'p2c_sha1s': df.groupby('project').size(),
                      'retrieved': commit_data.groupby('project').size()})
check['missing'] = check['p2c_sha1s'] - check['retrieved']
check.join(data_source.set_index('project'))

,p2c_sha1s,retrieved,missing,source
project,,,,
alanwilter_acpype,919,665,254,git (WoC unavailable)
aura-healthcare_hrv-analysis,177,168,9,git (WoC unavailable)
dakep_pense-rpkg,754,728,26,git (WoC unavailable)
declaredesign_estimatr,1423,1297,126,git (WoC unavailable)
igsio_openigtlinkio,692,592,100,git (WoC unavailable)
ipeagit_geobr,4942,4695,247,git (WoC unavailable)
juliaopt_cplex.jl,1159,567,592,git (WoC unavailable)
rrrlw_ripserr,429,314,115,git (WoC unavailable)
rwth-i6_returnn,16730,12167,4563,git (WoC unavailable)


## Step 3: save `gkhot_project_summary.csv` (semicolon separated)
Columns: project_wocid; commit_sha1; author; time (author unix time); commit message

In [6]:
summary = commit_data[['project', 'commit', 'author', 'author_time', 'message']].copy()
summary.columns = ['project_wocid', 'commit_sha1', 'author', 'time', 'commit message']
summary = summary.sort_values(['project_wocid', 'time']).reset_index(drop=True)
summary.to_csv(f'{netid}_project_summary.csv', sep=';', index=False)
print(len(summary), 'rows written')
summary.head()

27515 rows written


,project_wocid,commit_sha1,author,time,commit message
0,alanwilter_acpype,608ae68dc351ecc98167dab85276e5a0eb872c1d,alanwilter <alanwilter@gmail.com>,1508499658,Initial commit
1,alanwilter_acpype,d7bbde004584e0d74bd10d00c0017f80d69eaf6f,alanwilter <alanwilter@gmail.com>,1508503256,First commit\n\nmoving to GitHub
2,alanwilter_acpype,448d19813a22d0b1e103f33276aeff7a1a22771f,Alan Silva <alanwilter@gmail.com>,1508504686,First commit
3,alanwilter_acpype,61189931531497f29a36a8aadb092aa45bc79a16,Alan Silva <3899850+alanwilter@users.noreply.g...,1508505756,First commit
4,alanwilter_acpype,26dc7f6713e64aa21a6817df7b82af6c511f2ede,lkagami <luciano.kagami@ufrgs.br>,1508621447,Add files via upload


## Step 4: WoC statistics per project
Number of commits, number of distinct authors, min and max author time.

In [7]:
woc_stats = summary.groupby('project_wocid').agg(
    ncommits=('commit_sha1', 'nunique'),
    nauthors=('author', 'nunique'),
    min_time=('time', 'min'),
    max_time=('time', 'max')).reset_index()
woc_stats['from'] = pd.to_datetime(woc_stats['min_time'], unit='s').dt.strftime('%Y-%m-%d')
woc_stats['to'] = pd.to_datetime(woc_stats['max_time'], unit='s').dt.strftime('%Y-%m-%d')
woc_stats

,project_wocid,ncommits,nauthors,min_time,max_time,from,to
0,alanwilter_acpype,665,18,1508499658,1737363721,2017-10-20,2025-01-20
1,aura-healthcare_hrv-analysis,168,13,1536830840,1731336181,2018-09-13,2024-11-11
2,dakep_pense-rpkg,728,12,1454647156,1723317018,2016-02-05,2024-08-10
3,declaredesign_estimatr,1297,26,1490333150,1737585506,2017-03-24,2025-01-22
4,igsio_openigtlinkio,592,35,1302128240,1700082731,2011-04-06,2023-11-15
5,ipeagit_geobr,4695,46,1551275115,1737490732,2019-02-27,2025-01-21
6,juliaopt_cplex.jl,567,57,1381612383,1736114812,2013-10-12,2025-01-05
7,rrrlw_ripserr,314,16,1518701474,1728570660,2018-02-15,2024-10-10
8,rwth-i6_returnn,12167,159,1389707616,1738332966,2014-01-14,2025-01-31
9,sheffieldml_gpy,6322,153,1354122175,1736965367,2012-11-28,2025-01-15


## Step 5: GitHub statistics (stars, forks, last commit date on default branch)
Uses the GitHub REST API; set `GITHUB_TOKEN` to avoid the anonymous rate limit.

In [8]:
headers = {'Accept': 'application/vnd.github+json'}
if os.environ.get('GITHUB_TOKEN'):
    headers['Authorization'] = 'Bearer ' + os.environ['GITHUB_TOKEN']

gh_rows = []
for _, r in assign.iterrows():
    full = r['GH'].rstrip('/').split('github.com/')[1]
    repo = requests.get(f'https://api.github.com/repos/{full}', headers=headers).json()
    last = requests.get(f'https://api.github.com/repos/{full}/commits',
                        params={'per_page': 1}, headers=headers).json()
    gh_rows.append({'project_wocid': r['WoC'], 'GH': r['GH'],
                    'nstars': repo['stargazers_count'], 'nforks': repo['forks_count'],
                    'archived': repo['archived'],
                    'lastGHCommitDate': last[0]['commit']['committer']['date'][:10]})
    time.sleep(1)
gh_stats = pd.DataFrame(gh_rows)
gh_stats

,project_wocid,GH,nstars,nforks,archived,lastGHCommitDate
0,alanwilter_acpype,https://github.com/alanwilter/acpype,263,61,False,2026-09-24
1,sheffieldml_gpy,https://github.com/SheffieldML/GPy,2169,571,False,2026-10-07
2,rrrlw_ripserr,https://github.com/rrrlw/ripserr,12,7,False,2026-10-01
3,dakep_pense-rpkg,https://github.com/dakep/pense-rpkg,5,2,False,2026-01-27
4,declaredesign_estimatr,https://github.com/DeclareDesign/estimatr,135,22,False,2026-09-21
5,igsio_openigtlinkio,https://github.com/IGSIO/OpenIGTLinkIO,14,26,False,2025-07-18
6,aura-healthcare_hrv-analysis,https://github.com/Aura-healthcare/hrv-analysis,460,111,False,2026-06-02
7,juliaopt_cplex.jl,https://github.com/JuliaOpt/CPLEX.jl,139,62,False,2026-07-24
8,rwth-i6_returnn,https://github.com/rwth-i6/returnn,377,136,False,2026-10-07
9,ipeagit_geobr,https://github.com/ipeaGIT/geobr,959,131,False,2026-09-27


In [9]:
part1 = woc_stats.merge(gh_stats, on='project_wocid')
part1.to_csv('cache/part1_stats.csv', index=False)
print(part1[['project_wocid', 'ncommits', 'nauthors', 'from', 'to',
             'nstars', 'nforks', 'lastGHCommitDate']].to_markdown(index=False))

| project_wocid                |   ncommits |   nauthors | from       | to         |   nstars |   nforks | lastGHCommitDate   |
|:-----------------------------|-----------:|-----------:|:-----------|:-----------|---------:|---------:|:-------------------|
| alanwilter_acpype            |        665 |         18 | 2017-10-20 | 2025-01-20 |      263 |       61 | 2026-09-24         |
| aura-healthcare_hrv-analysis |        168 |         13 | 2018-09-13 | 2024-11-11 |      460 |      111 | 2026-06-02         |
| dakep_pense-rpkg             |        728 |         12 | 2016-02-05 | 2024-08-10 |        5 |        2 | 2026-01-27         |
| declaredesign_estimatr       |       1297 |         26 | 2017-03-24 | 2025-01-22 |      135 |       22 | 2026-09-21         |
| igsio_openigtlinkio          |        592 |         35 | 2011-04-06 | 2023-11-15 |       14 |       26 | 2025-07-18         |
| ipeagit_geobr                |       4695 |         46 | 2019-02-27 | 2025-01-21 |      959 |      131

## Commit data per project

From the commits retrieved above (git history up to 2025-01-31 because worldofcode.org was down, see Step 2). Min/max time are author times.

| Project | # commits | # authors | Min time | Max time |
|---|---|---|---|---|
| alanwilter_acpype | 665 | 18 | 1508499658 (2017-10-20) | 1737363721 (2025-01-20) |
| aura-healthcare_hrv-analysis | 168 | 13 | 1536830840 (2018-09-13) | 1731336181 (2024-11-11) |
| dakep_pense-rpkg | 728 | 12 | 1454647156 (2016-02-05) | 1723317018 (2024-08-10) |
| declaredesign_estimatr | 1297 | 26 | 1490333150 (2017-03-24) | 1737585506 (2025-01-22) |
| igsio_openigtlinkio | 592 | 35 | 1302128240 (2011-04-06) | 1700082731 (2023-11-15) |
| ipeagit_geobr | 4695 | 46 | 1551275115 (2019-02-27) | 1737490732 (2025-01-21) |
| juliaopt_cplex.jl | 567 | 57 | 1381612383 (2013-10-12) | 1736114812 (2025-01-05) |
| rrrlw_ripserr | 314 | 16 | 1518701474 (2018-02-15) | 1728570660 (2024-10-10) |
| rwth-i6_returnn | 12167 | 159 | 1389707616 (2014-01-14) | 1738332966 (2025-01-31) |
| sheffieldml_gpy | 6322 | 153 | 1354122175 (2012-11-28) | 1736965367 (2025-01-15) |

## GitHub data (collected 2026-10-07)

| Project | GitHub repo | Stars | Forks | Last commit date |
|---|---|---|---|---|
| alanwilter_acpype | alanwilter/acpype | 263 | 61 | 2026-09-24 |
| aura-healthcare_hrv-analysis | Aura-healthcare/hrv-analysis | 460 | 111 | 2026-06-02 |
| dakep_pense-rpkg | dakep/pense-rpkg | 5 | 2 | 2026-01-27 |
| declaredesign_estimatr | DeclareDesign/estimatr | 135 | 22 | 2026-09-21 |
| igsio_openigtlinkio | IGSIO/OpenIGTLinkIO | 14 | 26 | 2025-07-18 |
| ipeagit_geobr | ipeaGIT/geobr | 959 | 131 | 2026-09-27 |
| juliaopt_cplex.jl | JuliaOpt/CPLEX.jl | 139 | 62 | 2026-07-24 |
| rrrlw_ripserr | rrrlw/ripserr | 12 | 7 | 2026-10-01 |
| rwth-i6_returnn | rwth-i6/returnn | 377 | 136 | 2026-10-07 |
| sheffieldml_gpy | SheffieldML/GPy | 2169 | 571 | 2026-10-07 |